# Grammar-constrained decoding

Option 1 of the final project. The model turns an English sentence into a signup line. The baseline is the same run with the grammar mask turned off.

A legal line looks like `name=maya grade=10 club=robotics`.

A line is semantically correct when it parses and the name, grade, and club match the sentence. A legal line with the wrong field counts as wrong meaning. A line that does not parse counts as invalid.

In [1]:
import os
import warnings

os.environ["TQDM_DISABLE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", message="IProgress not found")

import demo

print("model:", demo.MODEL_NAME)
print("decoding: greedy (do_sample=False)")
print("max_new_tokens:", demo.MAX_NEW_TOKENS)
print("seed:", demo.SEED)
print("dtype: float32")
print("problems:", len(demo.PROBLEMS))
print()
print("system prompt:")
print(demo.SYSTEM)
print()
print("user prompt form:")
print("Write the signup line for this sentence: <sentence>")
print()
print("sentences:")
for sentence, name, grade, club in demo.PROBLEMS:
    print(f"- {sentence}")
    print(f"  gold: {demo.gold_line(name, grade, club)}")


model: Qwen/Qwen2.5-0.5B-Instruct
decoding: greedy (do_sample=False)
max_new_tokens: 32
seed: 0
dtype: float32
problems: 10

system prompt:
You write exactly one signup line and nothing else. The form is name=<word> grade=<number> club=<word>. Use lowercase letters only. The club is one word and does not include the word club. No quotes and no extra words.

user prompt form:
Write the signup line for this sentence: <sentence>

sentences:
- Maya is in grade 10 and wants the robotics club.
  gold: name=maya grade=10 club=robotics
- Sam is in grade 11 and wants the chess club.
  gold: name=sam grade=11 club=chess
- Ana is in grade 9 and wants the art club.
  gold: name=ana grade=9 club=art
- Leo is in grade 12 and wants the soccer club.
  gold: name=leo grade=12 club=soccer
- Nina is in grade 10 and wants the band club.
  gold: name=nina grade=10 club=band
- Omar is in grade 11 and wants the drama club.
  gold: name=omar grade=11 club=drama
- Priya is in grade 9 and wants the math club.
 

## Held fixed

Both conditions use the model, the prompts, greedy decoding, `max_new_tokens`, and the seed printed above. The grammar mask is the only change.

The next cell runs all 10 sentences. It loads the model, so the first run needs the model files (about 1 GB) and then takes about half a minute.

In [2]:
demo.main()


Grammar-constrained decoding demo
model: Qwen/Qwen2.5-0.5B-Instruct
decoding: greedy (do_sample=False)
max_new_tokens: 32
seed: 0
problems: 10
held fixed: model, prompts, greedy decoding, max_new_tokens, seed
only difference: grammar mask on or off
loading model...


grammar tokens: 41972



sentence: Maya is in grade 10 and wants the robotics club.
  gold: name=maya grade=10 club=robotics
  baseline:    'Maya is in grade 10 and wants the robotics club.'  invalid  0.7s
  constrained: 'name=maya grade=10 club=robotics'  correct  1.4s



sentence: Sam is in grade 11 and wants the chess club.
  gold: name=sam grade=11 club=chess
  baseline:    'Sam is in grade 11 and wants the chess club.'  invalid  0.7s
  constrained: 'name=smith grade=11 club=chess'  wrong_meaning  1.3s



sentence: Ana is in grade 9 and wants the art club.
  gold: name=ana grade=9 club=art
  baseline:    'Ana is in grade 9 and wants the art club.'  invalid  0.6s
  constrained: 'name=ana grade=9 club=art'  correct  1.1s



sentence: Leo is in grade 12 and wants the soccer club.
  gold: name=leo grade=12 club=soccer
  baseline:    'Leo is in grade 12 and wants the soccer club.'  invalid  0.7s
  constrained: 'name=leo grade=12 club=soccer'  correct  1.2s



... (2 outputs left out)

sentence: Priya is in grade 9 and wants the math club.
  gold: name=priya grade=9 club=math
  baseline:    'Priya is in grade 9 and wants the math club.'  invalid  0.7s
  constrained: 'name=priya grade=9 club=math'  correct  1.2s



sentence: Chris is in grade 12 and wants the debate club.
  gold: name=chris grade=12 club=debate
  baseline:    'Chris is in grade 12 and wants the debate club.'  invalid  0.9s
  constrained: 'name=chris grade=12 club=dbattle'  wrong_meaning  1.6s



sentence: Elena is in grade 10 and wants the dance club.
  gold: name=elena grade=10 club=dance
  baseline:    'Elena is in grade 10 and wants the dance club.'  invalid  0.7s
  constrained: 'name=elena grade=10 club=dance'  correct  1.3s



sentence: Jordan is in grade 11 and wants the coding club.
  gold: name=jordan grade=11 club=coding
  baseline:    'Jordan is in grade 11 and wants the coding club.'  invalid  0.7s
  constrained: 'name=jonathan grade=11 club=coding'  wrong_meaning  1.3s

Summary
  baseline    correct 0/10
              wrong_meaning 0/10
              invalid 10/10
  constrained correct 6/10
              wrong_meaning 4/10
              invalid 0/10
  average time baseline: 0.7s
  average time constrained: 1.3s

NUMBER: constrained semantic accuracy 6/10
NUMBER: baseline semantic accuracy 0/10
